# Go Beyond: True D=3 / Tetrahedron Contagion

This notebook studies true tetrahedron reinforcement for synthetic and empirical/import-ready complexes.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

CWD = Path.cwd().resolve()
candidates = [CWD, CWD.parent, CWD / 'Go_beyond', CWD.parent / 'Go_beyond']
ROOT = next(path for path in candidates if (path / 'core').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.go_beyond_experiments import run_prevalence_scan, run_time_evolution_scan, save_pickle
from core.go_beyond_generators import generate_family_complex
from core.go_beyond_import import import_sociopatterns_complex
from core.go_beyond_plotstyle import apply_style, ensure_output_dirs
from core.go_beyond_results import parse_results

apply_style()
FIGURES_DIR, RESULTS_DIR = ensure_output_dirs()
DATASET_PATH = ROOT.parent / 'Data' / 'Sociopatterns' / 'thr_data_random' / 'random_5_0.8min_cliques_Thiers13.json'

In [ ]:
PROFILE = 'smoke_test'  # change to 'production' for full runs

RUN_PROFILES = {
    'smoke_test': {
        'N': 180,
        't_max': 500,
        'n_simulations': 5,
        'lambda1s': np.linspace(0.2, 1.8, 18),
        'seed_pct': 5,
        'empirical_t_max': 350,
        'empirical_n_simulations': 4,
        'time_t_max': 200,
    },
    'production': {
        'N': 2000,
        't_max': 4000,
        'n_simulations': 80,
        'lambda1s': np.linspace(0.2, 2.2, 30),
        'seed_pct': 5,
        'empirical_t_max': 4000,
        'empirical_n_simulations': 80,
        'time_t_max': 4000,
    },
}

settings = RUN_PROFILES[PROFILE]
N = settings['N']
MU = 0.05
T_MAX = settings['t_max']
N_SIMS = settings['n_simulations']
LAMBDA1S = settings['lambda1s']
LAMBDA2_TARGET = 0.8
LAMBDA3_LIST = [0.0, 0.6, 1.2]
SEED_PCT = settings['seed_pct']

In [ ]:
synthetic_complex = generate_family_complex(
    family='er',
    mode='controlled',
    n_nodes=N,
    max_dimension=3,
    seed=31,
    k1=16,
    k2=4,
    k3=1.5,
)

synthetic_runs = {}
for lambda3_target in LAMBDA3_LIST:
    results = run_prevalence_scan(
        synthetic_complex,
        lambda1s=LAMBDA1S,
        lambda2_target=LAMBDA2_TARGET,
        lambda3_target=lambda3_target,
        seed_pct=SEED_PCT,
        t_max=T_MAX,
        mu=MU,
        n_sims=N_SIMS,
    )
    synthetic_runs[lambda3_target] = {
        'results': results,
        'summary': parse_results(results, cut=False),
    }

save_pickle(RESULTS_DIR / f'd3_synthetic_{PROFILE}.pkl', {'complex': synthetic_complex, 'runs': synthetic_runs})

In [ ]:
fig = plt.figure(figsize=(4.8, 3.4))
ax = plt.subplot(111)
colors = {0.0: 'black', 0.6: 'gray', 1.2: 'darkorange'}
markers = {0.0: 'o', 0.6: '^', 1.2: 's'}

for lambda3_target, payload in synthetic_runs.items():
    summary = payload['summary']
    ax.plot(
        LAMBDA1S,
        summary['avg_rhos'],
        '-',
        marker=markers[lambda3_target],
        color=colors[lambda3_target],
        mfc='white',
        ms=6,
        linewidth=1.2,
        label=fr'$\\lambda_3={lambda3_target}$',
    )
    ax.errorbar(LAMBDA1S, summary['avg_rhos'], yerr=summary['sem_rhos'], fmt='none', color=colors[lambda3_target], alpha=0.45, capsize=2, elinewidth=0.8)

ax.set_xlabel(r'Rescaled infectivity, $\lambda$', size=12)
ax.set_ylabel(r'Density of infected nodes, $\rho^{*}$', size=12)
ax.set_xlim(0.2, 1.8)
ax.set_ylim(-0.01, 0.9)
ax.legend(fontsize=9, loc='upper left', handlelength=1.1, handletextpad=0.4)
ax.set_title(f'Synthetic D=3 tetrahedron study ({PROFILE})', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'd3_synthetic_prevalence_{PROFILE}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if not DATASET_PATH.exists():
    raise FileNotFoundError(DATASET_PATH)

empirical_d2 = import_sociopatterns_complex(DATASET_PATH, max_dimension=2, seed=5)
empirical_d3 = import_sociopatterns_complex(DATASET_PATH, max_dimension=3, seed=5)

empirical_runs = {}
for label, complex_data, lambda3_target in [
    ('Imported D=2', empirical_d2, 0.0),
    ('Imported D=3', empirical_d3, 0.8),
]:
    results = run_prevalence_scan(
        complex_data,
        lambda1s=LAMBDA1S,
        lambda2_target=LAMBDA2_TARGET,
        lambda3_target=lambda3_target,
        seed_pct=SEED_PCT,
        t_max=settings['empirical_t_max'],
        mu=MU,
        n_sims=settings['empirical_n_simulations'],
    )
    empirical_runs[label] = {
        'complex_data': complex_data,
        'summary': parse_results(results, cut=False),
        'results': results,
    }

save_pickle(RESULTS_DIR / f'd3_empirical_example_{PROFILE}.pkl', empirical_runs)

In [ ]:
fig = plt.figure(figsize=(4.8, 3.4))
ax = plt.subplot(111)
emp_styles = {
    'Imported D=2': dict(marker='o', color='black', mfc='white'),
    'Imported D=3': dict(marker='s', color='cornflowerblue', mfc='white'),
}

for label, payload in empirical_runs.items():
    summary = payload['summary']
    sty = emp_styles[label]
    ax.plot(LAMBDA1S, summary['avg_rhos'], '-', linewidth=1.2, ms=6, **sty, label=label)
    ax.errorbar(LAMBDA1S, summary['avg_rhos'], yerr=summary['sem_rhos'], fmt='none', color=sty['color'], alpha=0.45, capsize=2, elinewidth=0.8)

ax.set_xlabel(r'Rescaled infectivity, $\lambda$', size=12)
ax.set_ylabel(r'Density of infected nodes, $\rho^{*}$', size=12)
ax.set_xlim(0.2, 1.8)
ax.set_ylim(-0.01, 0.9)
ax.legend(fontsize=9, loc='upper left', handlelength=1.1, handletextpad=0.4)
ax.set_title(f'Imported simplices: D=2 vs D=3 ({PROFILE})', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'd3_imported_comparison_{PROFILE}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
rho0_list = np.linspace(0.0, 1.0, 7)
time_histories = run_time_evolution_scan(
    synthetic_complex,
    lambda_target=0.85,
    lambda2_target=LAMBDA2_TARGET,
    lambda3_target=1.2,
    rho0_list=rho0_list,
    t_max=settings['time_t_max'],
    mu=MU,
)

save_pickle(RESULTS_DIR / f'd3_time_evolution_{PROFILE}.pkl', time_histories)

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 3.6))
colors = plt.cm.viridis(np.linspace(0.1, 0.9, len(time_histories)))
for color, payload in zip(colors, time_histories):
    ax.plot(range(len(payload['history'])), payload['history'], '-', color=color, lw=1.2, label=fr'$\\rho_0={payload["rho0"]:.2f}$')

ax.set_xlabel('Time steps', fontsize=12)
ax.set_ylabel(r'Density of infected nodes $\rho(t)$', fontsize=12)
ax.set_ylim(-0.02, 1.02)
ax.set_title(f'Synthetic D=3 time evolution ({PROFILE})', fontsize=11)
ax.legend(fontsize=8, loc='upper right', handlelength=1.2, labelspacing=0.2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / f'd3_time_evolution_{PROFILE}.png', dpi=150, bbox_inches='tight')
plt.show()